In [ ]:
# IMPORTS

from __future__ import annotations
from transformers import AutoTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

C:\Users\Fcomm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class LLM:
    def __init__(self):
        model_name = "google/flan-t5-small"
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def ask(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt")
        outputs = self.model.generate(**inputs, max_length=50)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

# create instance
llm = LLM()

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1925.43it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------------------------------------------------
# Step 0: Load the raw model and tokenizer directly
# ---------------------------------------------------------
model_id = "google/flan-t5-small"
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# The Raw Input State
grid_state = "- Ahead: Wall\n- Left: Floor\n- Right: Wall"

# ---------------------------------------------------------
# Step 1: The Evaluator Agent
# ---------------------------------------------------------
prompt_1 = f"""Task: Translate Wall to Blocked and Floor to Clear.
State:
{grid_state}
Translation:
"""

# Convert text to model tokens
inputs_1 = tokenizer(prompt_1, return_tensors="pt")

# Generate the output tokens (using max_new_tokens and do_sample=False for temperature=0.0 equivalent)
outputs_1 = model.generate(**inputs_1, max_new_tokens=15, do_sample=False)

# Decode tokens back into readable text
eval_output = tokenizer.decode(outputs_1[0], skip_special_tokens=True)
print(f"--- Agent 1 (Evaluator) Output ---\n{eval_output}\n")

# ---------------------------------------------------------
# Step 2: The Actor Agent
# ---------------------------------------------------------
prompt_2 = f"""Task: Pick the direction that is Clear. 
Options: left, right, forward.
Checks:
{eval_output}
Direction:
"""

inputs_2 = tokenizer(prompt_2, return_tensors="pt")
outputs_2 = model.generate(**inputs_2, max_new_tokens=2, do_sample=False)

final_action = tokenizer.decode(outputs_2[0], skip_special_tokens=True)
print(f"--- Agent 2 (Actor) Output ---\n{final_action}")

Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 16368.18it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


--- Agent 1 (Evaluator) Output ---
Task: Translate the wall to Blocked and Floor to Clear. State

--- Agent 2 (Actor) Output ---
Clear


In [4]:
#Gemini Pro
prompt = """
Context: You are an agent navigating a grid. You must not move into a Wall. You must move to a Floor. 
Current State:
- Ahead: Wall
- Left: Floor
- Right: Wall
- Diagonal Left: Floor
- Diagonal Right: Wall

Question: Based on the rules and current state, which action should you take? Choose from: left, right, forward. Answer with ONLY one word.

Answer:
"""
response = llm.ask(prompt)
print(response)

left


In [5]:
prompt = """
Context: You are an agent navigating a grid. You must not move into a Wall. You must move to a Floor. 

Current State:
- Ahead (1 step): Wall
- Ahead (2 steps): Floor
- Left: Wall
- Right: Floor
- Diagonal Left: Floor
- Diagonal Right: Wall

Question: Based on the rules and current state, which action should you take? Choose from: left, right, forward. Answer with ONLY one word.

Answer:
    """
response = llm.ask(prompt)
print(response)

left


In [6]:
prompt = """
Context: You are an agent navigating a grid. You must not move into a Wall. You must move to a Floor. 

Current State:
- Ahead (1 step): Floor
- Ahead (2 steps): Wall
- Left: Wall
- Right: Wall
- Diagonal Left: Floor
- Diagonal Right: Floor

Question: Based on the rules and current state, which action should you take? Choose from: left, right, forward. Answer with ONLY one word.

Answer:
    """
response = llm.ask(prompt)
print(response)

left


In [ ]:
prompt = """
Task: Grid Navigation. Follow the strict evaluation format exactly.
Valid Actions: move_forward, move_left, move_right, turn_left, turn_right.
Rule: You CANNOT move into a Wall. You CAN move into a Floor.

### Example 1
State:
- Ahead: Floor
- Left: Wall
- Right: Wall
- Diagonal Left: Wall
- Diagonal Right: Wall

Evaluation:
- Ahead is Floor -> move_forward is True
- Left is Wall -> move_left is False
- Right is Wall -> move_right is False
Action: move_forward

### Example 2
State:
- Ahead: Wall
- Left: Wall
- Right: Floor
- Diagonal Left: Wall
- Diagonal Right: Floor

Evaluation:
- Ahead is Wall -> move_forward is False
- Left is Wall -> move_left is False
- Right is Floor -> move_right is True
Action: move_right

### Target
State:
- Ahead: Wall
- Left: Floor
- Right: Wall
- Diagonal Left: Wall
- Diagonal Right: Floor

Evaluation:
    """
response = llm.ask(prompt)
print(response)

- Ahead is Wall -> move_right is False - Left is Wall -> move_right is False - Right is Wall -> move_right is True


In [8]:
#Originality.ai
prompt = """
You are a navigation agent operating in a 2D grid-based world.
Your ultimate mission is to reach the "goal tile".
You MUST do so via the most optimal sequence of moves.

**Your perceptive field is limited to 6 tiles around you:**
- **Ahead:** the tile directly in front of you
- **Left:** the tile directly to your left
- **Right:** the tile directly to your right
- **Diagonal Left:** the tile diagonally forward-left
- **Diagonal Right:** the tile diagonally forward-right
- You can also **turn in-place**, **right** or **left** (90 degrees). This can help you e.g. go backwards, by turning in-place twice then moving forward.

**Movement rules:**
- You CANNOT enter wall tiles
- You CAN move onto floor tiles

---

**Reasoning Protocol — follow each step in order:**

0. Build, in your working memory, a map of the place as you explore it.
1. **Assess immediate moves:** Which of the valid actions are physically possible right now?
2. **Project one step further:** For each possible move, what does the diagonal and secondary tile information suggest about what lies beyond?
3. **Evaluate goal proximity:** Which passable move most plausibly advances toward an unknown goal, given no walls are blocking that corridor?
4. **Select optimal action:** Choose the single action with the best forward progress potential while avoiding immediate dead ends.

**Output format:**
- State your reasoning concisely through each step
- End with a single clearly labeled action: **Action: `[action]`**

Apply a greedy-but-aware strategy: prioritize immediate progress toward the goal while using diagonal tile data to avoid committing to paths that dead-end within the next move.

---

**Current Environment State:**

| Direction | Tile |
|---|---|
| Ahead | Wall |
| Left | Floor |
| Right | Wall |
| Diagonal Left | Wall |
| Diagonal Right | Floor |
"""
response = llm.ask(prompt)
print(response)

--- | Ahead is the wall, the right is the floor, and the right is the floor. The direction is the tile, and the tile is the tile diagonally forward-left.**


In [9]:
prompt = """
SYSTEM DIRECTIVE: You are an autonomous navigation drone (Unit 74-B) operating in a subterranean grid complex.
OBJECTIVE: Calculate the optimal trajectory to maximize survivability while progressing toward an unknown target coordinate.

ENVIRONMENTAL DEFINITIONS:
- Floor [F]: Safe to traverse. Costs 1 movement point.
- Wall [W]: Impassable solid matter. Fatal collision risk.
- Hazard [H]: Electrified grid. Destroys drone on contact.

SENSORY INPUT (3x3 Local Matrix):
[ Forward-Left: W ]  [ Forward: F ]      [ Forward-Right: W ]
[ Left: F ]          [ CURRENT: D ]      [ Right: H ]

EXTENDED RADAR (Depth 2):
- Forward x2: W
- Left x2: F
- Right x2: H

ANALYTICAL PROTOCOL:
Step 1: Parse the local matrix and identify all immediately adjacent [F] tiles.
Step 2: Cross-reference identified [F] tiles with the Extended Radar to ensure they do not lead into a dead end (W) or hazard (H).
Step 3: Eliminate any paths that require moving backward unless all forward/lateral paths are blocked.
Step 4: Select the most mathematically sound directional vector (forward, backward, left, right).
"""
response = llm.ask(prompt)
print(response)

Step 3: Select the most mathematically sound directional vector (forward, backward, left, right)


In [33]:
prompt = """
Environment:
- Left: wall
- Right: wall

Question: Which direction is blocked?

Answer:
"""
response = llm.ask(prompt)
print(response)

Right


In [ ]:
prompt = """
**Rules**
You are a navigation agent operating in a 2D grid-based world.
Your ultimate mission is to reach the "goal tile".
You MUST do so via the most optimal sequence of moves.

Your perceptive field is limited to 3 tiles around you:
- Ahead: the tile directly in front of you
- Left: the tile directly to your left
- Right: the tile directly to your right

**Movement rules:**
- You CANNOT enter wall tiles
- You CAN move onto floor tiles

---

**Reasoning Protocol — follow each step in order:**
1. Build, in your working memory, a map of the place as you explore it.
2. Assess immediate moves: Which of the valid actions are physically possible right now?
3. Evaluate goal proximity: Which passable move most plausibly advances toward an unknown goal, given no walls are blocking that corridor?
4. Select optimal action: Choose the single action with the best forward progress potential while avoiding immediate dead ends.

**Output format:**
- If you find moving forward the most optimal move reply foward.
- If you find moving left the most optimal move reply left.
- If you find moving right the most optimal move reply right.

---

**Current Environment State:**
This is what your current perceptive field sees:

Ahead is Wall 
Left is Floor 
Right is Wall 
"""
response = llm.ask(prompt)
print(response)

Right is Wall ---
